<a href="https://colab.research.google.com/github/sara-iqbal/LIVE-Creator-Growth-Segmentation/blob/main/tiktok_live_creator_growth_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TikTok LIVE Creator Growth & Segmentation

This notebook builds an end-to-end analysis for a regional LIVE Operations team:

1. **Synthetic dataset** of LIVE creator behaviour (watch time, gifting, follower growth, content vertical)
2. **RFM-style segmentation** (Recency / Frequency / Monetary) with KMeans clustering
3. **Predictive model** that flags "high-potential" creators early, using only leading indicators
4. **Leading vs. lagging indicator framework** tied to illustrative OKRs
5. **Content-vertical opportunity analysis** — where to focus creator recruitment
6. **Growth funnel** from registered creator to Star Creator
7. **Export** of `dashboard_data.json` + `creators_synthetic_dataset.csv` for the HTML dashboard

> The dataset is fully synthetic (randomly generated with realistic distributions) — no real
> creator or platform data is used. Run all cells top to bottom in Google Colab.


## 0. Setup

In [ ]:
!pip install -q scikit-learn pandas numpy

import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

pd.set_option("display.width", 140)
np.random.seed(42)
N = 2500


## 1. Synthetic dataset

We simulate 2,500 LIVE creators across 6 content verticals and 6 regions, with a 28-day
activity window. Metrics are generated so that **tenure** (creator maturity) and
**content vertical** both plausibly influence watch time, follower growth, and gifting —
mirroring real LIVE dynamics without using any real platform data.

In [ ]:
verticals = ["Gaming", "Beauty & Fashion", "Music & Dance", "Education/Talk",
             "Comedy & Lifestyle", "Cooking & Food"]
vertical_weights = [0.16, 0.20, 0.22, 0.12, 0.18, 0.12]
regions = ["UK", "Germany", "France", "Spain", "Italy", "Poland"]

df = pd.DataFrame({
    "creator_id": [f"C{100000+i}" for i in range(N)],
    "content_vertical": np.random.choice(verticals, N, p=vertical_weights),
    "region": np.random.choice(regions, N),
    "tenure_days": np.random.randint(7, 730, N),
})

tenure_factor = np.clip(df["tenure_days"] / 365, 0.1, 2.0)
vertical_gift_mult = df["content_vertical"].map({
    "Gaming": 1.3, "Beauty & Fashion": 1.1, "Music & Dance": 1.4,
    "Education/Talk": 0.7, "Comedy & Lifestyle": 1.0, "Cooking & Food": 0.9})
vertical_watch_mult = df["content_vertical"].map({
    "Gaming": 1.2, "Beauty & Fashion": 1.0, "Music & Dance": 1.3,
    "Education/Talk": 1.1, "Comedy & Lifestyle": 1.15, "Cooking & Food": 0.95})

df["live_sessions_28d"] = np.random.poisson(8, N) + (tenure_factor * 2).astype(int)
df["avg_session_min"] = np.clip(np.random.normal(35, 15, N) * vertical_watch_mult, 5, 180)
df["avg_concurrent_viewers"] = np.clip(np.random.gamma(2, 40, N) * vertical_watch_mult * (0.5 + 0.5*tenure_factor), 5, None)
df["total_watch_time_hrs_28d"] = (df["live_sessions_28d"] * df["avg_session_min"] * df["avg_concurrent_viewers"] / 60).round(1)

df["new_followers_28d"] = np.clip(np.random.gamma(2, 25, N) * (0.4 + 0.6*tenure_factor), 0, None).astype(int)
df["follower_base"] = np.clip((df["new_followers_28d"] * np.random.uniform(3, 25, N) * tenure_factor), 50, None).astype(int)
df["follower_growth_rate_28d"] = (df["new_followers_28d"] / df["follower_base"].clip(lower=1)).round(4)

df["unique_gifters_28d"] = np.clip(np.random.poisson(6, N) * vertical_gift_mult * (0.3 + 0.7*tenure_factor), 0, None).astype(int)
df["diamonds_28d"] = np.clip(np.random.gamma(2, 400, N) * vertical_gift_mult * (0.4 + 0.6*tenure_factor), 0, None).round(0)
df["repeat_gifter_rate"] = np.clip(np.random.beta(2, 5, N) + 0.05*tenure_factor, 0, 1).round(3)

df["comments_per_session"] = np.clip(np.random.gamma(3, 12, N) * vertical_watch_mult, 0, None).round(1)
df["community_guideline_strikes_90d"] = np.random.choice([0,0,0,0,1,2], N, p=[0.55,0.2,0.1,0.08,0.05,0.02])
df["est_monthly_revenue_usd"] = (df["diamonds_28d"] * 0.005).round(2)
df["days_since_last_live"] = np.clip(np.random.exponential(4, N).astype(int), 0, 45)

df.head()


## 2. RFM-style scoring

Classic RFM (Recency / Frequency / Monetary) analysis, adapted for LIVE:

- **Recency** → days since the creator last went LIVE (lower is better)
- **Frequency** → LIVE sessions in the last 28 days
- **Monetary** → estimated monthly revenue from gifting

Each dimension is scored 1–5 by quintile, then summed into an `RFM_score` (3–15).

In [ ]:
def score_quantile(series, ascending=True, bins=5):
    ranks = series.rank(method="first", ascending=ascending)
    return pd.qcut(ranks, bins, labels=list(range(1, bins+1))).astype(int)

df["R_score"] = score_quantile(df["days_since_last_live"], ascending=False)
df["F_score"] = score_quantile(df["live_sessions_28d"], ascending=True)
df["M_score"] = score_quantile(df["est_monthly_revenue_usd"], ascending=True)
df["RFM_score"] = df["R_score"] + df["F_score"] + df["M_score"]

df[["creator_id","R_score","F_score","M_score","RFM_score"]].head()


## 3. Creator segmentation (KMeans clustering)

We cluster on RFM scores plus follower growth, reach, and gifting loyalty to find
natural creator segments, then label each cluster from its behavioural profile.

In [ ]:
cluster_features = ["R_score", "F_score", "M_score", "follower_growth_rate_28d",
                     "avg_concurrent_viewers", "repeat_gifter_rate"]
X_scaled = StandardScaler().fit_transform(df[cluster_features])

km = KMeans(n_clusters=5, n_init=10, random_state=42)
df["cluster"] = km.fit_predict(X_scaled)

df.groupby("cluster")[cluster_features + ["est_monthly_revenue_usd","live_sessions_28d","tenure_days"]].mean().round(2)


Based on the profile of each cluster (high monetary + frequency = stars, high growth +
low tenure = rising talent, low recency = at risk, etc.), we assign human-readable labels.
**Re-run the cell above first if you change the random seed or feature set — cluster
numbers can shuffle, so re-check the profile before trusting this mapping.**

In [ ]:
cluster_label_map = {
    4: "Star Creators",
    0: "High-Reach Veterans",
    3: "Steady Core",
    2: "Rising Talent",
    1: "At Risk / Dormant",
}
df["segment"] = df["cluster"].map(cluster_label_map)

segment_summary = (df.groupby("segment")
                    .agg(creators=("creator_id", "count"),
                         avg_revenue=("est_monthly_revenue_usd", "mean"),
                         avg_growth_rate=("follower_growth_rate_28d", "mean"),
                         avg_viewers=("avg_concurrent_viewers", "mean"),
                         avg_sessions=("live_sessions_28d", "mean"),
                         avg_tenure=("tenure_days", "mean"))
                    .round(2)
                    .reset_index())
segment_summary


## 4. High-potential creator classifier

**Goal:** flag creators likely to break out in the next ~90 days *before* it shows up in
revenue — using only **leading indicators** (engagement momentum), not lagging ones
(current revenue/RFM, which are outcomes we're trying to predict ahead of).

The target label is a simulated "momentum score" built from growth rate, comments,
gifting breadth/loyalty and viewer reach, thresholded at the top 20% among
earlier-tenure creators. In a real deployment this label would instead be *actual*
future-90-day outcomes (e.g. "did this creator reach Star Creator status by day 90").

In [ ]:
def z(s):
    return (s - s.mean()) / s.std()

momentum_score = (
    z(df["follower_growth_rate_28d"]) * 1.4 +
    z(df["comments_per_session"]) * 1.0 +
    z(df["unique_gifters_28d"]) * 1.1 +
    z(df["repeat_gifter_rate"]) * 0.9 +
    z(df["avg_concurrent_viewers"]) * 0.8 -
    z(df["tenure_days"]) * 0.5 +
    np.random.normal(0, 0.8, N)
)
threshold = momentum_score.quantile(0.80)
df["high_potential"] = ((momentum_score >= threshold) & (df["tenure_days"] < 450)).astype(int)
df["high_potential"].value_counts()


In [ ]:
feature_cols = ["live_sessions_28d", "avg_session_min", "avg_concurrent_viewers",
                "follower_growth_rate_28d", "unique_gifters_28d", "repeat_gifter_rate",
                "comments_per_session", "tenure_days"]
X = pd.get_dummies(df[feature_cols + ["content_vertical"]], columns=["content_vertical"], drop_first=True)
y = df["high_potential"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                              class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(8)


In [ ]:
# Score every creator, produce the "predicted high-potential" watchlist for Ops
df["high_potential_proba"] = clf.predict_proba(X)[:, 1]
predicted_list = (df.sort_values("high_potential_proba", ascending=False)
                     .head(25)[["creator_id","content_vertical","region","segment",
                                "tenure_days","follower_growth_rate_28d",
                                "avg_concurrent_viewers","high_potential_proba"]]
                     .round(3))
predicted_list


## 5. Leading vs. lagging indicator framework

Ops teams need to know *what to watch this week* (leading) vs. *what tells us if we
succeeded* (lagging), mapped to a small set of illustrative OKRs.

In [ ]:
okr_framework = [
    {"okr": "Grow sustainable LIVE creator base",
     "leading_indicators": ["New creators completing 3+ sessions in first 14 days",
                             "Session frequency trend (this week vs last week)",
                             "Comments per session (early engagement signal)"],
     "lagging_indicators": ["28-day active creator count", "Creator churn rate (90 day)"]},
    {"okr": "Increase creator monetization health",
     "leading_indicators": ["Unique gifters per session", "Repeat gifter rate",
                             "Follower-to-viewer conversion rate"],
     "lagging_indicators": ["Diamonds / estimated revenue per creator",
                             "% of creators crossing monetization threshold"]},
    {"okr": "Identify and support high-potential creators early",
     "leading_indicators": ["Follower growth rate (28 day)", "Avg concurrent viewers trend",
                             "Engagement depth in first 60 days"],
     "lagging_indicators": ["Creator reaches Star Creators segment", "Time-to-monetization"]},
    {"okr": "Maintain a healthy, trustworthy creator ecosystem",
     "leading_indicators": ["Community guideline education completion",
                             "Early strike rate in first 30 days"],
     "lagging_indicators": ["Community guideline strikes (90 day)",
                             "Creator suspension / ban rate"]},
]
for row in okr_framework:
    print("OKR:", row["okr"])
    print("  Leading:", row["leading_indicators"])
    print("  Lagging:", row["lagging_indicators"])
    print()


## 6. Content vertical opportunity analysis

Where should creator recruitment focus? We score each vertical on growth rate,
revenue, share of high-potential creators, and how saturated (already crowded)
it already is.

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

vertical_summary = (df.groupby("content_vertical")
                     .agg(creators=("creator_id","count"),
                          avg_revenue=("est_monthly_revenue_usd","mean"),
                          avg_growth_rate=("follower_growth_rate_28d","mean"),
                          avg_viewers=("avg_concurrent_viewers","mean"),
                          high_potential_rate=("high_potential","mean"))
                     .round(3)
                     .reset_index())

vertical_summary["saturation"] = vertical_summary["creators"] / vertical_summary["creators"].sum()
vertical_summary["opportunity_score"] = (
    minmax(vertical_summary["avg_growth_rate"]) * 0.35 +
    minmax(vertical_summary["avg_revenue"]) * 0.25 +
    minmax(vertical_summary["high_potential_rate"]) * 0.25 -
    minmax(vertical_summary["saturation"]) * 0.15
).round(3)

vertical_summary.sort_values("opportunity_score", ascending=False)


## 7. Growth funnel

A simple funnel from "registered creator" down to "Star Creator", useful for
tracking where creators drop off.

In [ ]:
funnel = [
    {"stage": "Registered LIVE Creators", "count": int(N)},
    {"stage": "Active (>=1 session/28d)", "count": int((df["live_sessions_28d"] >= 1).sum())},
    {"stage": "Regular Streamers (>=8 sessions/28d)", "count": int((df["live_sessions_28d"] >= 8).sum())},
    {"stage": "Monetizing (diamonds > 0)", "count": int((df["diamonds_28d"] > 0).sum())},
    {"stage": "High-Value (M_score >= 4)", "count": int((df["M_score"] >= 4).sum())},
    {"stage": "Star Creators (segment)", "count": int((df["segment"] == "Star Creators").sum())},
]
pd.DataFrame(funnel)


## 8. Export dashboard data

Everything the HTML dashboard needs is written to `dashboard_data.json`
(plus the full dataset to `creators_synthetic_dataset.csv`). Download both from the
Colab file browser, or mount Google Drive, and drop `dashboard_data.json`'s contents
into `dashboard.html` (or host both files together) for the GitHub-hosted dashboard.

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True)
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred).tolist()

dashboard_data = {
    "meta": {
        "n_creators": int(N),
        "model_auc": round(float(auc), 3),
        "model_precision_high_potential": round(report["1"]["precision"], 3),
        "model_recall_high_potential": round(report["1"]["recall"], 3),
        "confusion_matrix": cm,
    },
    "segments": segment_summary.to_dict(orient="records"),
    "funnel": funnel,
    "verticals": vertical_summary.sort_values("opportunity_score", ascending=False).to_dict(orient="records"),
    "feature_importance": [{"feature": k, "importance": round(float(v), 4)} for k, v in importances.head(8).items()],
    "predicted_high_potential": predicted_list.to_dict(orient="records"),
    "okr_framework": okr_framework,
    "segment_scatter": (df.sample(400, random_state=1)
                          [["creator_id","segment","content_vertical",
                            "follower_growth_rate_28d","avg_concurrent_viewers",
                            "est_monthly_revenue_usd"]]
                          .round(3).to_dict(orient="records")),
}

with open("dashboard_data.json", "w") as f:
    json.dump(dashboard_data, f, indent=2)

df.to_csv("creators_synthetic_dataset.csv", index=False)
print("Saved dashboard_data.json and creators_synthetic_dataset.csv")


## 9. Quick sanity-check visuals (optional)

Matplotlib charts to eyeball the segmentation and funnel before trusting the export.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

segment_summary.plot(kind="bar", x="segment", y="creators", ax=axes[0], legend=False, color="#25F4EE")
axes[0].set_title("Creators per segment")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=30)

funnel_df = pd.DataFrame(funnel)
axes[1].barh(funnel_df["stage"], funnel_df["count"], color="#FE2C55")
axes[1].invert_yaxis()
axes[1].set_title("Growth funnel")

plt.tight_layout()
plt.show()
